# 03 — First features

Computes pre-fight rolling features on top of the de-biased fight table from
notebook 02, closing leakage traps #2 and #3 from `ROADMAP.md`: the raw
per-fight stats (strikes, takedowns, control time...) are outcomes of the
fight they're attached to, so they can't be used directly. This notebook
rolls them up per fighter, using only fights strictly before the one being
predicted, and attaches the static (time-invariant) attributes — age,
height, reach, stance — on top.

Weight is deliberately excluded: it changes across a fighter's career as
they move divisions, so a single joined value wouldn't be safe the way
reach/stance are — `weight_class` on the fight itself is the correct source
if weight is ever needed.

As with notebook 02, the actual transform lives in
`src/ufc_predictor/features/` (`rolling.py`, `static.py`) — this notebook
calls it and checks the result.

In [1]:
from ufc_predictor.data.io import load_processed, load_fighters, save_processed
from ufc_predictor.features.rolling import attach_prior_features
from ufc_predictor.features.static import attach_static_features

fights = load_processed("fights")
fighters = load_fighters()

featured = attach_prior_features(fights)
featured = attach_static_features(featured, fighters)
featured.shape

(7697, 170)

## Sanity checks

In [2]:
# no fights lost or duplicated by the merge
assert len(featured) == len(fights)

# a genuine debut fighter has no rolled-up history to draw on
assert featured["fighter_a_prior_is_debut"].sum() > 0
debut_rows = featured[featured["fighter_a_prior_is_debut"]]
assert debut_rows["fighter_a_prior_sig_strikes_career_avg"].isna().all()

any_debut_rate = (
    featured["fighter_a_prior_is_debut"] | featured["fighter_b_prior_is_debut"]
).mean()
print(f"fights involving at least one debutant: {any_debut_rate:.1%}")

fights involving at least one debutant: 26.1%


In [3]:
# leakage spot-check: Jim Miller's 3rd UFC fight (2009-03-07) should show a
# prior record of 2-0, not his eventual career record — this is the exact
# check that verified the module against a hand-computed example.
#
# he lands in the fighter_a slot for some fights and fighter_b for others
# (that's the point of the de-biasing in notebook 02), so both sides need
# checking, not just fighter_a.
jim_miller_id = 2500
mask = (featured["fighter_a_id"] == jim_miller_id) | (featured["fighter_b_id"] == jim_miller_id)
his_fights = featured[mask].sort_values("event_date")
his_3rd_fight = his_fights.iloc[2]
side = "fighter_a" if his_3rd_fight["fighter_a_id"] == jim_miller_id else "fighter_b"

prior_wins = his_3rd_fight[f"{side}_prior_wins"]
prior_losses = his_3rd_fight[f"{side}_prior_losses"]
print("date:", his_3rd_fight["event_date"].date())
print("prior_wins:", prior_wins)
print("prior_losses:", prior_losses)
assert prior_wins == 2
assert prior_losses == 0

date: 2009-03-07
prior_wins: 2.0
prior_losses: 0.0


In [4]:
# static feature sanity checks: row count preserved, diffs are simple
# subtraction, missingness matches what we already knew about the raw
# fighter table (reach ~45%, stance ~19%) rather than something worse
assert len(featured) == len(fights)

diff_pairs = {
    "diff_static_age": ("fighter_a_age_during", "fighter_b_age_during"),
    "diff_static_height_cm": ("fighter_a_height_cm", "fighter_b_height_cm"),
    "diff_static_reach_cm": ("fighter_a_reach_cm", "fighter_b_reach_cm"),
}
for diff_col, (a_col, b_col) in diff_pairs.items():
    assert featured[diff_col].equals(featured[a_col] - featured[b_col])

print("reach missing:", f"{featured['fighter_a_reach_cm'].isna().mean():.1%}")
print("stance missing:", f"{featured['fighter_a_stance'].isna().mean():.1%}")
print("age missing:", f"{featured['fighter_a_age_during'].isna().mean():.1%}")

reach missing: 7.5%
stance missing: 0.5%
age missing: 1.2%


## A look at the engineered features

In [5]:
diff_cols = [c for c in featured.columns if c.startswith("diff_prior_") or c.startswith("diff_static_")]
print(f"{len(diff_cols)} diff features")
featured[diff_cols].describe().T[["mean", "std", "min", "max"]]

37 diff features


,mean,std,min,max
diff_prior_fight_count,-0.001429,6.373275,-40.000000,36.000000
diff_prior_wins,-0.001429,4.012050,-24.000000,21.000000
diff_prior_losses,0.000000,2.787101,-16.000000,16.000000
diff_prior_win_rate,0.004764,0.312367,-1.000000,1.000000
diff_prior_streak,0.010913,2.304832,-16.000000,12.000000
diff_prior_days_since_last_fight,-2.403515,297.608400,-4047.000000,3179.000000
diff_prior_sig_strikes_career_avg,-0.094016,25.014846,-138.500000,164.750000
diff_prior_sig_strike_atts_career_avg,-0.353975,56.427858,-250.666667,401.400000
diff_prior_takedowns_career_avg,-0.020342,1.596162,-10.000000,10.000000
diff_prior_takedown_atts_career_avg,-0.031088,3.557116,-19.500000,30.000000


## Save

In [6]:
save_processed(featured, "fights_featured")
featured.head()

,weight_class,fights_url,event_url,event_date,fighter_a_age_during,fighter_b_age_during,fighter_a_body_strike_atts,fighter_b_body_strike_atts,fighter_a_body_strikes,fighter_b_body_strikes,...,diff_prior_knockdowns_absorbed_last3_avg,diff_prior_submissions_absorbed_last3_avg,diff_prior_ctrl_time_absorbed_last3_avg,fighter_a_reach_cm,fighter_a_stance,fighter_b_reach_cm,fighter_b_stance,diff_static_age,diff_static_height_cm,diff_static_reach_cm
0,Open Weight Bout,http://www.ufcstats.com/fight-details/17ee4caf...,http://www.ufcstats.com/event-details/a6a9ab5a...,1994-03-11,NaN,NaN,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,Southpaw,NaN,-17.78,NaN
1,Open Weight Bout,http://www.ufcstats.com/fight-details/5655639b...,http://www.ufcstats.com/event-details/a6a9ab5a...,1994-03-11,24.0,27.0,0,0,0,0,...,NaN,NaN,NaN,NaN,Southpaw,NaN,Southpaw,-3.0,-5.08,NaN
2,Open Weight Bout,http://www.ufcstats.com/fight-details/aa161c71...,http://www.ufcstats.com/event-details/a6a9ab5a...,1994-03-11,NaN,27.0,0,0,0,0,...,0.0,0.0,NaN,NaN,Southpaw,NaN,Southpaw,NaN,5.08,NaN
3,Open Weight Bout,http://www.ufcstats.com/fight-details/c53325be...,http://www.ufcstats.com/event-details/a6a9ab5a...,1994-03-11,NaN,24.0,0,2,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,Southpaw,NaN,2.54,NaN
4,Open Weight Bout,http://www.ufcstats.com/fight-details/8ea72cf8...,http://www.ufcstats.com/event-details/a6a9ab5a...,1994-03-11,NaN,30.0,0,1,0,1,...,0.0,0.0,NaN,NaN,Orthodox,NaN,Orthodox,NaN,-10.16,NaN


## Visualise the dataset

The full table is 172 columns, too wide to read directly. A curated view of
the most recent fights: who fought, the label, and the headline engineered
features.

In [7]:
readable_cols = [
    "event_date", "weight_class",
    "fighter_a_name", "fighter_b_name", "fighter_a_won",
    "diff_prior_win_rate", "diff_prior_streak",
    "diff_prior_sig_strikes_career_avg", "diff_prior_takedowns_career_avg",
    "diff_static_age", "diff_static_reach_cm", "diff_static_height_cm",
]

view = featured[readable_cols].sort_values("event_date", ascending=False)
view.head(20)

,event_date,weight_class,fighter_a_name,fighter_b_name,fighter_a_won,diff_prior_win_rate,diff_prior_streak,diff_prior_sig_strikes_career_avg,diff_prior_takedowns_career_avg,diff_static_age,diff_static_reach_cm,diff_static_height_cm
7696,2024-11-09,Welterweight Bout,Carlos Prates,Neil Magny,True,0.333333,4.0,-7.303030,-1.787879,-6.0,-5.08,-5.08
7691,2024-11-09,Women's Strawweight Bout,Gillian Robertson,Luana Pinheiro,True,0.047059,4.0,-12.764706,-0.152941,-2.0,2.54,7.62
7686,2024-11-09,Middleweight Bout,Dusko Todorovic,Mansur Abdul-Malik,False,NaN,-1.0,NaN,NaN,3.0,-12.70,-2.54
7687,2024-11-09,Middleweight Bout,Antonio Trocoli,Tresean Gore,False,-0.333333,-2.0,-9.000000,-1.666667,3.0,10.16,15.24
7689,2024-11-09,Bantamweight Bout,Da'Mon Blackshear,Cody Stamann,True,-0.138462,0.0,-11.338462,-0.015385,-5.0,20.32,10.16
7690,2024-11-09,Welterweight Bout,Charles Radtke,Matthew Semelsberger,True,0.166667,2.0,-18.366667,-0.466667,3.0,-7.62,-10.16
7688,2024-11-09,Bantamweight Bout,Cortavious Romious,Gaston Bolanos,False,NaN,1.0,NaN,NaN,-2.0,-2.54,-7.62
7692,2024-11-09,Women's Bantamweight Bout,Klaudia Sygula,Melissa Mullins,False,NaN,1.0,NaN,NaN,-8.0,2.54,2.54
7693,2024-11-09,Middleweight Bout,Reinier de Ridder,Gerald Meerschaert,True,NaN,-2.0,NaN,NaN,-2.0,2.54,7.62
7694,2024-11-09,Welterweight Bout,Zachary Scroggin,Elizeu Zaleski dos Santos,False,NaN,1.0,NaN,NaN,-9.0,2.54,10.16
